# Developing a simple Knowledge Graph in Kanren


Consider the following familiar situation. A University department teaches students, who are enrolled on different modules.

* Students enroll on a single course.
* Each course requires a set of modules
* Each module is taught by a Professor.
* Students pass individual modules

We therefore have the following entities:

* `Course`
* `Module`
* `Student`
* `Professor`

and predicates

* `Requires(course, module)`
* `Enrolled(student, programme)`
* `Teaches(professor, module)`
* `Passed(student, module)`

and ontology (in graphical form)

![Simple ontology of academic programme structure](figures/FirstOrderLogicGraphSchema.png)


Kanren provides us with mechanisms for directly representing these, but these mechanisms are limited. We can specify the type of relation/predicate, and we can create instances of those relation, but there is no mechanism for specifying the ontology, or the types of the entities. These are represented as strings

In [1]:
from kanren import facts, Relation, run, var

Passed = Relation()
Requires = Relation()
Teaches = Relation()
Enrolled = Relation()


Let's add some facts

In [2]:
facts(Requires,("AI","ECS8050"),("AI","ECS8051"),("AI","ECS8052"))
facts(Enrolled,("Alice","AI"),("Bob","AI"),("Chen","AI"))
facts(Teaches,("Diana","ECS8050"),("Edward","ECS8051"),("Faye","ECS8052"))
# An alternative way to do this which will be helpful when we load from file. The * unpacks f into its elements.
f = [("Alice","ECS8050"),("Alice","ECS8051"),("Alice","ECS8052"),("Bob","ECS8050"),("Bob","ECS8052"),("Chen","ECS8050"),("Chen","ECS8051"),("Chen","ECS8052")]
facts(Passed, *f)

Let's run a couple of simple queries over this
* Who teaches ECS8051?
* Who is enrolled on the MSc in AI?
* What modules has Bob passed

In [3]:
x = var()
y = run(0,x,Requires(x, "ECS8051"))
print(y)
y = run(0,x,Enrolled(x, "AI"))
print(y)
y = run(0,x,Passed("Bob",x))
print(y)


('AI',)
('Bob', 'Alice', 'Chen')
('ECS8052', 'ECS8050')


Typing in all the data is very labourious so let's load it from a pre-prepared CSV file

In [4]:
import csv
from kanren import facts, Relation
d = dict()
with open('2.03 SimpleGraph.csv') as csvfile:
    triples = csv.reader(csvfile)
    # Put the triples in a dictionary keyed by the relations
    for triple in triples:
        if triple[1] not in d.keys():
            d[triple[1]] = [(triple[0],triple[2])]
        else:
            d[triple[1]].append((triple[0],triple[2]))

# Now we create the facts. This is a little tricky.
# We first add the relations to the dictionary of global variables.
# We then pass the list of facts in using * to unpack the list into its members
for k in d.keys():
    globals()[k] = Relation()
    facts(globals()[k],*d[k])

print(globals().keys())

FileNotFoundError: [Errno 2] No such file or directory: '2.03 SimpleGraph.csv'

## Querying the graph

Now we have the graph, we can write some queries to extract information from it. Let's start with a simple query - find all the students on a programme.

In [ ]:
# Placeholder for code
x = var()
y=var()
run(0, x, Requires(x,'ECS8051'))

Now let's find all the the students who take a particular module. To do this, we need to find all of the programms that require the module, and then all of the students on those programs.

In [ ]:
y = var()
run(0, x, Enrolled(x,y), Requires(y, "ECS8050"))

Using this, we can infer a new relationship `TakesModule(student, module)`. Let's do this and then add in this new relationship.

In [ ]:
# First we get a list of all of the modules required by any program. We can use an unbound variable as a wildcard.
modules = set(run(0, y, Requires(x,y)))

TakesModule = Relation()
f = []
# Now we use the query above to get all the students on each of these module
for m in modules:
    students = run(0, x, Enrolled(x,y), Requires(y, m))
    pairs = [(s,m) for s in students]
    f = f + pairs

facts(TakesModule, *f)

Let's query this

In [ ]:
run(0, x, TakesModule(x,"ECS8058"))

Note that there are a few ways in which we could have done this. Can you come up with another one?